In [1]:
import pandas as pd
import numpy as np

In [ ]:
#Carga del dataframe original
df=pd.read_csv('../data/openb_pod_list_cpu0.csv')

In [ ]:
#Guardado del dataframe procesado
df.to_csv('../data/processed_data.csv', index=False)

In [ ]:
#Carga el dataframe procesado
df=pd.read_csv('../data/processed_data.csv')

In [25]:
print(f"Cantidad de Columnas: {df.columns.nunique()}\nCantidad de filas: {df.shape[0]}\nCantidad de nulos: {df.isnull().sum().sum()}\nNombres:\n {df.columns}")

Cantidad de Columnas: 11
Cantidad de filas: 4334
Cantidad de nulos: 0
Nombres:
 Index(['name', 'cpu_milli', 'memory_mib', 'num_gpu', 'gpu_milli', 'qos',
       'pod_phase', 'creation_time', 'deletion_time', 'scheduled_time',
       'duration_time'],
      dtype='str')


In [15]:
df.isnull().sum()

name                 0
cpu_milli            0
memory_mib           0
num_gpu              0
gpu_milli            0
gpu_spec          7064
qos                  0
pod_phase            0
creation_time        0
deletion_time        0
scheduled_time     861
dtype: int64

In [16]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7064 entries, 0 to 7063
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   name            7064 non-null   str    
 1   cpu_milli       7064 non-null   int64  
 2   memory_mib      7064 non-null   int64  
 3   num_gpu         7064 non-null   int64  
 4   gpu_milli       7064 non-null   int64  
 5   gpu_spec        0 non-null      float64
 6   qos             7064 non-null   str    
 7   pod_phase       7064 non-null   str    
 8   creation_time   7064 non-null   int64  
 9   deletion_time   7064 non-null   int64  
 10  scheduled_time  6203 non-null   float64
dtypes: float64(2), int64(6), str(3)
memory usage: 607.2 KB


In [17]:
df.tail(5)

,name,cpu_milli,memory_mib,num_gpu,gpu_milli,gpu_spec,qos,pod_phase,creation_time,deletion_time,scheduled_time
7059,openb-pod-7059,16000,32768,1,1000,NaN,LS,Succeeded,12897160,12900786,12897161.0
7060,openb-pod-7060,11400,77824,1,1000,NaN,LS,Running,12897659,12898203,12897692.0
7061,openb-pod-7061,6000,18432,1,460,NaN,LS,Running,12898342,12900034,12898342.0
7062,openb-pod-7062,3152,5600,1,590,NaN,BE,Failed,12900505,12900526,12900505.0
7063,openb-pod-7063,3152,5600,1,590,NaN,BE,Failed,12901761,12901792,12901762.0


In [ ]:
#Eliminamos la columna que no tiene valores ya que solo aportaría ruido al modelo.
df.drop(columns=['gpu_spec'], inplace=True)

In [19]:
#Elinamos todos los registros que tienen como estatus "Pending" ya que no aportan metricas de información para el modelo
df.dropna(subset=['scheduled_time'], inplace=True)


In [ ]:
#Eliminamos todos los registros que tienen como estatus "Failed" ya que no aportan metricas de información para el modelo
df=df[df['pod_phase']!='Failed']

In [23]:
#       .:Feature Engineering:.

#Obtenemos la duración real del pod
df['duration_time']=df['deletion_time']-df['creation_time']

#Volvemos númericas las variables categorícas de la columna qos con el metodo del one hot encoding
df_fe=pd.get_dummies(df,columns=['qos'], drop_first=True,dtype=int)

#Eliminamos todas las columnas que no aportan valor matemático al modelo
columnas_a_eleminar=['name','pod_phase','deletion_time','creation_time','scheduled_time']
df_modelo=df_fe.drop(columns=columnas_a_eleminar)

#Separamos las variables que fungirán como características (x) y la variable que fungirá como target (y)
X=df_modelo.drop(columns=['cpu_milli'])
y=df_modelo['cpu_milli']

In [31]:
print(f"\t~~~~Estadisticas del df_model~~~~\nCantidad de columnas: {df_modelo.columns.nunique()}\nCantidad de filas: {df_modelo.shape[0]}\nColumnas: {df_modelo.columns}")

	~~~~Estadisticas del df_model~~~~
Cantidad de columnas: 8
Cantidad de filas: 4334
Columnas: Index(['cpu_milli', 'memory_mib', 'num_gpu', 'gpu_milli', 'duration_time',
       'qos_Burstable', 'qos_Guaranteed', 'qos_LS'],
      dtype='str')


In [ ]:
#Guardamos el dataframe que se utilizará para entrenar el modelo
df_modelo.to_csv('../data/df_modelo.csv', index=False)

In [ ]:
#Cargamos el dataframe que se utilizará para entrenar el modelo
df_modelo=pd.read_csv('../data/df_modelo.csv')